# Conflicts-with Repair Analyzer

In [ ]:
import pandas as pd

# You can read the already calculated repairs file, or read the one generated on 't-box repairs analyzer' folder with the script 't-box_repairs_analyzer.py'
filename = "../../datasets/conflicts_with_reqPropVal_repairs.csv"
filename = "../../datasets/conflicts_with_onlyProp_repairs.csv"

# Set the 'only' variable based on the filename
only = 'only' in filename.lower()

df1 = pd.read_csv(filename)
df1

- The cell below counts different types of basic T-box repairs generated with the relational database:

In [ ]:
# T-box changes coming from the relational db
if only:
    print(len(df1[(df1['C_deleted'] == True)]))
    print(len(df1[(df1['C_deprecated'] == True)]))
    print(len(df1[(df1['CQ_added_exception'] == True)]))
    print(len(df1[(df1['CQ_deleted_property'] == True)]))
else:
    print(len(df1[(df1['C_deleted'] == True)]))
    print(len(df1[(df1['C_deprecated'] == True)]))
    print(len(df1[(df1['CQ_added_exception'] == True)]))
    print(len(df1[(df1['CQ_deleted_value'] == True)]))

- check for base statement deletions:

In [ ]:
import requests
import xml.etree.ElementTree as ET

def isRemoved(subject, property):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    property = property.replace('http://www.wikidata.org/entity/','http://www.wikidata.org/prop/direct/')
    # SPARQL query
    query = f"ASK {{ <{subject}> <{property}> [] }}"

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)

    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'false'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None


In [ ]:
df1['S_deleted'] = None

In [ ]:
from tqdm import tqdm

# Wrap the dataframe with tqdm to show progress
for index, row in tqdm(df1.iterrows(), total=len(df1), desc="Processing rows"):
    # Extract subject and property without the prefix "http://www.wikidata.org/entity/"
        
    subject = row['subject']
    property = row['property']
    
    # Call isRemoved function
    removed = isRemoved(subject, property)

    # Update S_deleted column
    df1.at[index, 'S_deleted'] = removed

In [ ]:
len(df1[(df1['S_deleted'] == True)])

In [ ]:
import requests
import xml.etree.ElementTree as ET

def isRemovedWithObj(subject, property, obj):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    property = property.replace('http://www.wikidata.org/entity/','http://www.wikidata.org/prop/direct/')
    # SPARQL query
    query = f"ASK {{ <{subject}> <{property}> <{obj}> }}"

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)

    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            return boolean_element.text.lower() == 'false'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

- check for forbidden statement deletions:

In [ ]:
if only:
    df1['Sc_deleted_only'] = None

In [ ]:
if only:
    from tqdm import tqdm
    
    # Wrap the dataframe with tqdm to show progress
    for index, row in tqdm(df1.iterrows(), total=len(df1), desc="Processing rows"):
        # Extract subject and property without the prefix "http://www.wikidata.org/entity/"
            
        subject = row['subject']
        property = row['PC']
        
        # Call isRemoved function
        removed = isRemoved(subject, property)
    
        df1.at[index, 'Sc_deleted_only'] = removed

In [ ]:
if only:
    print(len(df1[(df1['Sc_deleted_only'] == True)]))

- testing context statement value deletion:

In [ ]:
if not only:
    df1['Sc_deleted_reqPropVal'] = None

In [ ]:
if not only:
    from tqdm import tqdm
    
    # Wrap the dataframe with tqdm to show progress
    for index, row in tqdm(df1.iterrows(), total=len(df1), desc="Processing rows"):
        if row['VC'].startswith('http'):
            obj = row['VC']
    
            wdtStmtRemoved = isRemovedWithObj(row['subject'], row['PC'], obj)
    
            df1.at[index, 'Sc_deleted_reqPropVal'] = wdtStmtRemoved

In [ ]:
if not only:
    print(len(df1[(df1['Sc_deleted_reqPropVal'] == True)]))

- test for object replacement:

In [ ]:
import requests
import xml.etree.ElementTree as ET

def getSQStatement(subject, property, object):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2019"

    # Prepare the property
    pid = property.replace('http://www.wikidata.org/prop/direct/', '')

    # SPARQL query
    query = f"""
    PREFIX p: <http://www.wikidata.org/prop/>
    PREFIX ps: <http://www.wikidata.org/prop/statement/>
    SELECT ?SQ {{
        <{subject}> p:{pid} ?SQ.
        ?SQ ps:{pid} <{object}>
    }}
    """

    # URL encode the query
    encoded_query = requests.utils.quote(query)

    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"

    # Send HTTP GET request
    headers = {"Accept": "application/sparql-results+xml"}

    response = requests.get(url, headers=headers)

    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        root = ET.fromstring(response.text)
        namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}

        # Collect all uri and literal elements
        uris = [uri.text for uri in root.findall('.//ns:uri', namespace)]
        literals = [lit.text for lit in root.findall('.//ns:literal', namespace)]

        results = uris + literals  # Combine both types of results

        return results if results else None
    else:
        print("Error:", response.text)
        return None

def getObjectsOfStatementPS(subject, property):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # Prepare the property
    pid = property.replace('http://www.wikidata.org/prop/direct/', '')

    # SPARQL query
    query = f"""
    SELECT ?o {{
        <{subject}> ps:{pid} ?o
    }}
    """

    # URL encode the query
    encoded_query = requests.utils.quote(query)

    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"

    # Send HTTP GET request
    headers = {"Accept": "application/sparql-results+xml"}

    response = requests.get(url, headers=headers)

    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        root = ET.fromstring(response.text)
        namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}

        # Collect all uri and literal elements
        uris = [uri.text for uri in root.findall('.//ns:uri', namespace)]
        literals = [lit.text for lit in root.findall('.//ns:literal', namespace)]

        results = uris + literals  # Combine both types of results

        return results if results else None
    else:
        print("Error:", response.text)
        return None

In [ ]:
if not only:
    df1['Sr_replacement'] = None

In [ ]:
if not only:
    from tqdm import tqdm
    import os

    for index, row in tqdm(df1.iterrows(), total=len(df1)):
        if row['Sr_replacement'] is None:
            CQ_list = getSQStatement(row['subject'], row['PC'], row['VC'])
            
            if CQ_list is None:
                print(f"CQ_list is None for subject: {row['subject']}")
                df1.at[index, 'Sr_replacement'] = False 
                continue
            
            obj_list = getObjectsOfStatementPS(CQ_list[0], row['PC'])
            if obj_list is None or row['VC'] in obj_list:
                df1.at[index, 'Sr_replacement'] = False
            else:
                df1.at[index, 'Sr_replacement'] = True


In [ ]:
if not only:
    print(len(df1[(df1['Sr_replacement'] == True)]))

In [ ]:
if only:
    rows_to_drop = df1[
        (df1['C_deleted'] == False) &
        (df1['C_deprecated'] == False) &
        (df1['CQ_added_exception'] == False) &
        (df1['CQ_deleted_property'] == False) &
        (df1['S_deleted'] == False) 
    ]
else:
    rows_to_drop = df1[
        (df1['C_deleted'] == False) &
        (df1['C_deprecated'] == False) &
        (df1['CQ_added_exception'] == False) &
        (df1['CQ_deleted_value'] == False) &
        (df1['S_deleted'] == False) & 
        (df1['Sc_deleted_reqPropVal'] == False) &
        (df1['Sr_replacement'] == False) 
    ]

In [ ]:
df1_filtered = df1.drop(rows_to_drop.index)

In [ ]:
if only:
    df1_filtered.to_csv('final_conflicts-with-only-req-prop_repairs.csv', index=False)
else:
    df1_filtered.to_csv('final_conflicts-with-req-prop-val_repairs.csv', index=False)  